# Test your trained model on the full PDF (Colab)

Same logic as `test_model.py`, but running on Colab so you're not limited by local CPU speed across all 499 pages.

No GPU needed for this one (inference is fast either way) but leave it on if you already switched it on for training.

In [ ]:
# 1. Install dependencies (poppler comes from apt, not pip, on Linux)
!apt-get -qq install poppler-utils
!pip install -q ultralytics pdf2image opencv-python-headless

In [ ]:
# 2a. Upload your PDF (pick just index.cfm.pdf in the dialog)
from google.colab import files
print("Choose your PDF file (e.g. index.cfm.pdf)")
uploaded_pdf = files.upload()
print("Uploaded:", list(uploaded_pdf.keys()))

In [ ]:
# 2b. Upload your trained weights (pick just best_colab.pt in the dialog)
print("Choose your model weights file (e.g. best_colab.pt)")
uploaded_model = files.upload()
print("Uploaded:", list(uploaded_model.keys()))

In [ ]:
# 3. Settings - edit these, then run
PDF_PATH   = "index.cfm.pdf"       # change if you uploaded a different PDF
MODEL_PATH = "best_colab.pt"       # change if you uploaded a different weights file
START_PAGE = 4
MAX_PAGES  = 496                    # 496 = the rest of a 499-page PDF; lower this for a quick sample
DPI        = 150
CONF       = 0.25
OUT_DIR    = "test_results"

In [ ]:
# 4. Run detection across the page range (mirrors test_model.py's test_pdf())
import os
import cv2
import numpy as np
from ultralytics import YOLO
from pdf2image import convert_from_path, pdfinfo_from_path

CLASS_NAMES = [
    "logo", "text_logo", "serial_number", "description", "agent",
    "applicant", "class", "date", "international_registration_date",
]
CLASS_COLORS = {
    "logo": (0, 0, 255), "text_logo": (180, 0, 255), "serial_number": (255, 0, 0),
    "description": (0, 170, 0), "agent": (0, 136, 255), "applicant": (255, 0, 170),
    "class": (255, 170, 0), "date": (47, 52, 74), "international_registration_date": (19, 69, 139),
}

def draw_results(img, results, conf_threshold):
    annotated = img.copy()
    boxes = results[0].boxes
    detections = []
    if boxes is None or len(boxes) == 0:
        return annotated, detections
    for box in boxes:
        conf = float(box.conf[0])
        if conf < conf_threshold:
            continue
        cls_id = int(box.cls[0])
        cls_name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f"class_{cls_id}"
        color = CLASS_COLORS.get(cls_name, (200, 200, 200))
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        label = f"{cls_name} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(annotated, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
        cv2.putText(annotated, label, (x1 + 2, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        detections.append({"class": cls_name, "conf": round(conf, 3)})
    return annotated, detections

model = YOLO(MODEL_PATH)
info = pdfinfo_from_path(PDF_PATH)
total_pages = info["Pages"]
end_page = min(START_PAGE + MAX_PAGES - 1, total_pages)
print(f"PDF has {total_pages} pages. Processing {START_PAGE}-{end_page} at {DPI} DPI...")

pages = convert_from_path(PDF_PATH, dpi=DPI, first_page=START_PAGE, last_page=end_page)
os.makedirs(OUT_DIR, exist_ok=True)

total_detections = 0
for i, page in enumerate(pages):
    page_num = START_PAGE + i
    img = cv2.cvtColor(np.array(page), cv2.COLOR_RGB2BGR)
    results = model(img, conf=CONF, verbose=False)
    annotated, detections = draw_results(img, results, CONF)
    total_detections += len(detections)
    cv2.imwrite(os.path.join(OUT_DIR, f"page_{page_num:04d}_result.jpg"), annotated)
    if (i + 1) % 20 == 0 or i == len(pages) - 1:
        print(f"  [{i+1}/{len(pages)}] pages processed, {total_detections} detections so far")

print(f"\nDone - {total_detections} total detections across {len(pages)} pages")
print(f"Results saved in: {OUT_DIR}/")

In [ ]:
# 5. Zip and download the annotated results
import shutil
from google.colab import files

shutil.make_archive("test_results", "zip", "test_results")
files.download("test_results.zip")